# Regression: Learning Functions from Data
Regression is the task of learning a function $f(x) = y$ from labeled data $(x_i, y_i)$. The challenge: we don't know the true underlying function, and we have noise in our observations.

We'll explore three dimensions:
1. **Loss functions**: How do we quantify "good fit"?
2. **Linear regression**: The simplest case with optimal closed-form solution
3. **Non-linear functions**: Polynomials, exponentials, and when simple models fail

The overarching lesson: more complex models aren't always better; overfitting is a greater danger than underfitting.

<details>

<summary>Regression vs Classification</summary>
Regression: Predict continuous values (house price, temperature, stock returns).<br><br>
Classification: Predict discrete categories (spam/not-spam, dog/cat/bird). Regression can be converted to classification via thresholding, and vice versa.
</details>

## The Problem: Predicting House Prices
You have 100 houses with square footage and sale prices. Can you predict price from size?

Generate synthetic data that follows a linear trend plus noise:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures

np.random.seed(42)

# Generate data: price ~ 100 * size + noise
n_samples = 100
X = np.random.uniform(800, 4000, n_samples)  # square feet
y_true = 100 * X + 50000  # Base: $100/sq ft + $50k
noise = np.random.normal(0, 50000, n_samples)  # Gaussian noise
y = y_true + noise  # Observed prices (noisy)

# Sort for visualization
idx = np.argsort(X)
X = X[idx]
y = y[idx]
y_true = y_true[idx]

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, s=50, edgecolors='black', linewidth=0.5, label='Observed (noisy)')
plt.plot(X, y_true, 'r--', linewidth=2, label='True underlying trend')
plt.xlabel('Square Footage')
plt.ylabel('Price ($)')
plt.title('House Price Data: Can we recover the true trend?')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Dataset: {n_samples} houses")
print(f"Features: square footage in [{X.min():.0f}, {X.max():.0f}]")
print(f"Prices: ${y.min():.0f} to ${y.max():.0f}")

## Loss Functions: Measuring "Goodness of Fit"
A loss function quantifies the error between predicted $\hat{y}$ and observed $y$.

**Mean Squared Error (MSE)**:
$$L_{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

Pros: Differentiable (good for optimization), penalizes large errors quadratically

Cons: Sensitive to outliers (squared term amplifies big errors)

**Mean Absolute Error (MAE)**:
$$L_{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$

Pros: Robust to outliers (linear penalty)

Cons: Not differentiable at 0, harder to optimize

Let's visualize the difference:

In [ ]:
# Create sample predictions with varying errors
errors = np.linspace(-10, 10, 100)
mse_loss = errors ** 2
mae_loss = np.abs(errors)
huber_loss = np.where(np.abs(errors) <= 2, 0.5 * errors**2, 2 * np.abs(errors) - 2)  # Huber loss

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(errors, mse_loss, linewidth=2, label='MSE')
plt.plot(errors, mae_loss, linewidth=2, label='MAE')
plt.xlabel('Error (prediction - truth)')
plt.ylabel('Loss')
plt.title('Loss Functions: How errors are penalized')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(errors, mse_loss, linewidth=2, label='MSE')
plt.plot(errors, huber_loss, linewidth=2, label='Huber (hybrid)')
plt.xlabel('Error (prediction - truth)')
plt.ylabel('Loss')
plt.title('MSE vs Huber: Robustness to outliers')
plt.legend()
plt.grid(alpha=0.3)

# Show effect on a single outlier
plt.subplot(1, 3, 3)
test_errors = np.array([1, 2, 5, 10])
mse_vals = test_errors ** 2
mae_vals = np.abs(test_errors)

x_pos = np.arange(len(test_errors))
width = 0.35
plt.bar(x_pos - width/2, mse_vals, width, label='MSE', alpha=0.8)
plt.bar(x_pos + width/2, mae_vals, width, label='MAE', alpha=0.8)
plt.xlabel('Error magnitude')
plt.ylabel('Total Loss')
plt.title('MSE penalizes outliers more heavily')
plt.xticks(x_pos, [f"{e}" for e in test_errors])
plt.legend()
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Why MSE is popular:")
print("  - Squaring makes optimization smooth (derivatives exist)")
print("  - Large errors are heavily penalized (usually desired)")
print("\nWhy MAE is robust:")
print("  - A single outlier doesn't dominate (linear penalty)")
print("  - Closer to human intuition (average dollar error)")

<details>

<summary>Why is differentiability important for loss functions?</summary>
Optimization algorithms (gradient descent, Newton's method) compute derivatives to find minima. MSE is smooth everywhere; MAE has a sharp corner at error=0. This makes MAE harder to optimize numerically, though modern solvers handle it.
</details>

## Linear Regression: The Simplest Case
Assume the true function is linear: $f(x) = wx + b$

Minimize MSE by solving the normal equations (analytic solution exists):
$$w = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}$$
$$b = \bar{y} - w \bar{x}$$

Let's implement linear regression from scratch and compare to sklearn:

In [ ]:
class SimpleLinearRegression:
    def __init__(self):
        self.w = None
        self.b = None
    
    def fit(self, X, y):
        """
        Fit linear model y = w*X + b using normal equations
        """
        n = len(X)
        
        # Compute means
        x_mean = np.mean(X)
        y_mean = np.mean(y)
        
        # Compute slope and intercept
        numerator = np.sum((X - x_mean) * (y - y_mean))
        denominator = np.sum((X - x_mean) ** 2)
        
        self.w = numerator / denominator
        self.b = y_mean - self.w * x_mean
        
        return self
    
    def predict(self, X):
        return self.w * X + self.b
    
    def score(self, X, y):
        """
        Return R^2 score (coefficient of determination)
        """
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

# Fit our custom model
custom_model = SimpleLinearRegression()
custom_model.fit(X, y)

# Fit sklearn model for comparison
sklearn_model = LinearRegression()
sklearn_model.fit(X.reshape(-1, 1), y)

# Predictions
y_pred_custom = custom_model.predict(X)
y_pred_sklearn = sklearn_model.predict(X.reshape(-1, 1)).flatten()

# Metrics
mse_custom = np.mean((y - y_pred_custom) ** 2)
mse_sklearn = np.mean((y - y_pred_sklearn) ** 2)
r2_custom = custom_model.score(X, y)
r2_sklearn = r2_score(y, y_pred_sklearn)

# Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.6, s=50, edgecolors='black', linewidth=0.5, label='Data')
plt.plot(X, y_true, 'r--', linewidth=2, label='True trend')
plt.plot(X, y_pred_custom, 'g-', linewidth=2, label=f'Linear fit (R²={r2_custom:.3f})')
plt.xlabel('Square Footage')
plt.ylabel('Price ($)')
plt.title('Linear Regression: Best-fit line')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y - y_pred_custom
plt.scatter(y_pred_custom, residuals, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Price ($)')
plt.ylabel('Residual ($)')
plt.title('Residual Plot: Should be random')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Custom Implementation:")
print(f"  w = {custom_model.w:.4f}, b = {custom_model.b:.2f}")
print(f"  MSE = {mse_custom:.2f}, R² = {r2_custom:.4f}")
print(f"\nSklearn Implementation:")
print(f"  w = {sklearn_model.coef_[0]:.4f}, b = {sklearn_model.intercept_:.2f}")
print(f"  MSE = {mse_sklearn:.2f}, R² = {r2_sklearn:.4f}")
print(f"\nInterpretation: Each additional sq ft increases price by ${custom_model.w:.2f}")

<details>

<summary>What is R² (coefficient of determination)?</summary>
R² measures the fraction of variance explained by the model. $R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$ where $SS_{res}$ is residual sum of squares and $SS_{tot}$ is total sum of squares. Values: $R^2 = 1$ (perfect fit), $R^2 = 0$ (model no better than mean), $R^2 < 0$ (worse than mean!).
</details>

## The Limitation: Linear Models on Non-Linear Data
Real data often isn't linear. Suppose prices follow a different pattern:

In [ ]:
# Generate non-linear data: price ~ sqrt(size) (diminishing returns)
X_nonlin = np.random.uniform(800, 4000, n_samples)
y_nonlin_true = 1000 * np.sqrt(X_nonlin)
noise = np.random.normal(0, 30000, n_samples)
y_nonlin = y_nonlin_true + noise

# Sort
idx = np.argsort(X_nonlin)
X_nonlin = X_nonlin[idx]
y_nonlin = y_nonlin[idx]
y_nonlin_true = y_nonlin_true[idx]

# Fit linear model
linear_model = SimpleLinearRegression()
linear_model.fit(X_nonlin, y_nonlin)
y_pred_linear = linear_model.predict(X_nonlin)

r2_linear = linear_model.score(X_nonlin, y_nonlin)

plt.figure(figsize=(10, 6))
plt.scatter(X_nonlin, y_nonlin, alpha=0.6, s=50, edgecolors='black', linewidth=0.5, label='Data')
plt.plot(X_nonlin, y_nonlin_true, 'r--', linewidth=2, label='True trend (√x)')
plt.plot(X_nonlin, y_pred_linear, 'g-', linewidth=2, label=f'Linear fit (R²={r2_linear:.3f})')
plt.xlabel('Square Footage')
plt.ylabel('Price ($)')
plt.title('Linear Regression on Non-Linear Data: Poor fit')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Linear fit on non-linear data: R² = {r2_linear:.4f}")
print("Notice: The linear model consistently under/over-predicts (not random residuals).")

## Non-Linear Functions: Polynomial Regression
Fit polynomial functions $f(x) = w_0 + w_1 x + w_2 x^2 + ... + w_d x^d$

This is still linear in the coefficients! We create polynomial features and use linear regression.

In [ ]:
degrees = [1, 2, 3, 5, 10]
models = {}

plt.figure(figsize=(16, 5))

for idx, degree in enumerate(degrees):
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X_nonlin.reshape(-1, 1))
    
    # Fit linear regression on polynomial features
    model = LinearRegression()
    model.fit(X_poly, y_nonlin)
    models[degree] = (model, poly)
    
    # Predictions on dense grid for smooth curve
    X_dense = np.linspace(X_nonlin.min(), X_nonlin.max(), 300)
    X_dense_poly = poly.transform(X_dense.reshape(-1, 1))
    y_pred_dense = model.predict(X_dense_poly)
    
    # Predictions on training data
    y_pred_train = model.predict(X_poly)
    r2_score_val = r2_score(y_nonlin, y_pred_train)
    
    # Plot
    ax = plt.subplot(1, 5, idx + 1)
    plt.scatter(X_nonlin, y_nonlin, alpha=0.4, s=30, edgecolors='black', linewidth=0.3)
    plt.plot(X_nonlin, y_nonlin_true, 'r--', linewidth=1.5, label='True')
    plt.plot(X_dense, y_pred_dense, 'g-', linewidth=2, label=f'Fit (R²={r2_score_val:.3f})')
    plt.xlabel('Size (sq ft)')
    if idx == 0:
        plt.ylabel('Price ($)')
    plt.title(f'Degree {degree}')
    plt.grid(alpha=0.3)
    if idx == 0:
        plt.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Observation: Higher degrees fit training data better, but...")

## The Trade-off: Bias vs Variance (Overfitting)
**Underfitting** (high bias): Model too simple, doesn't capture true pattern

**Overfitting** (high variance): Model too complex, memorizes noise instead of learning signal

Let's use train/test split to detect overfitting:

In [ ]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_nonlin, y_nonlin, test_size=0.3, random_state=42
)

# Fit models of increasing complexity
degrees = range(1, 12)
train_r2 = []
test_r2 = []
train_mse = []
test_mse = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train.reshape(-1, 1))
    X_test_poly = poly.transform(X_test.reshape(-1, 1))
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Evaluate
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)
    
    train_r2.append(r2_score(y_train, y_train_pred))
    test_r2.append(r2_score(y_test, y_test_pred))
    train_mse.append(mean_squared_error(y_train, y_train_pred))
    test_mse.append(mean_squared_error(y_test, y_test_pred))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(degrees, train_r2, 'o-', linewidth=2, markersize=8, label='Train R²')
plt.plot(degrees, test_r2, 's-', linewidth=2, markersize=8, label='Test R²')
plt.xlabel('Polynomial Degree')
plt.ylabel('R² Score')
plt.title('R²: Gap indicates overfitting')
plt.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='True complexity')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogy(degrees, train_mse, 'o-', linewidth=2, markersize=8, label='Train MSE')
plt.semilogy(degrees, test_mse, 's-', linewidth=2, markersize=8, label='Test MSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('MSE (log scale)')
plt.title('MSE: Divergence at high degrees')
plt.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='True complexity')
plt.legend()
plt.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"{'Degree':<8} {'Train R²':<10} {'Test R²':<10} {'Gap':<10}")
for deg, tr2, ter2 in zip(degrees, train_r2, test_r2):
    gap = tr2 - ter2
    print(f"{deg:<8} {tr2:<10.4f} {ter2:<10.4f} {gap:<10.4f}")

print("\nNotice: Beyond degree ~4, train R² keeps improving but test R² plateaus or worsens.")
print("This is OVERFITTING—the model memorizes training noise.")

<details>

<summary>How do we choose the right complexity?</summary>
**Cross-validation**: Split data into k folds, train on k-1, test on 1, repeat k times. Evaluates generalization without losing data.<br><br>
**Regularization**: Add penalty term to loss (e.g., Ridge = MSE + λ||w||²). Forces small weights, preventing overfitting.<br><br>
**Model selection**: Use validation set (not test) to compare models, pick the one with best validation performance.
</details>

## Different Function Classes: Beyond Polynomials
We can fit exponential, logarithmic, or other functional forms by transforming features:

In [ ]:
# Create data that follows different true functions
np.random.seed(42)
X_demo = np.linspace(1, 10, 50)

# Different underlying functions
y_linear = 2 * X_demo + 5 + np.random.normal(0, 1, 50)
y_exponential = 10 * np.exp(0.5 * X_demo) + np.random.normal(0, 50, 50)
y_logarithmic = 20 * np.log(X_demo) + np.random.normal(0, 2, 50)
y_power = 5 * X_demo**2 + np.random.normal(0, 5, 50)

functions = [
    ('Linear: y = w*x + b', y_linear, lambda x: x.reshape(-1, 1)),
    ('Exponential: y = w*e^x + b', y_exponential, lambda x: np.exp(x).reshape(-1, 1)),
    ('Logarithmic: y = w*ln(x) + b', y_logarithmic, lambda x: np.log(x).reshape(-1, 1)),
    ('Power: y = w*x^2 + b', y_power, lambda x: (x**2).reshape(-1, 1))
]

plt.figure(figsize=(14, 10))

for plot_idx, (title, y_data, transform) in enumerate(functions, 1):
    X_transformed = transform(X_demo)
    
    # Fit linear model on transformed features
    model = LinearRegression()
    model.fit(X_transformed, y_data)
    y_pred = model.predict(X_transformed)
    r2 = r2_score(y_data, y_pred)
    
    plt.subplot(2, 2, plot_idx)
    plt.scatter(X_demo, y_data, s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    # Plot fit curve
    if 'exp' in title.lower():
        X_smooth = np.linspace(X_demo.min(), X_demo.max(), 100)
        X_smooth_transformed = np.exp(X_smooth).reshape(-1, 1)
    elif 'log' in title.lower():
        X_smooth = np.linspace(X_demo.min(), X_demo.max(), 100)
        X_smooth_transformed = np.log(X_smooth).reshape(-1, 1)
    elif 'power' in title.lower():
        X_smooth = np.linspace(X_demo.min(), X_demo.max(), 100)
        X_smooth_transformed = (X_smooth**2).reshape(-1, 1)
    else:
        X_smooth = np.linspace(X_demo.min(), X_demo.max(), 100)
        X_smooth_transformed = X_smooth.reshape(-1, 1)
    
    y_smooth = model.predict(X_smooth_transformed)
    plt.plot(X_smooth, y_smooth, 'r-', linewidth=2, label=f'Fit (R²={r2:.3f})')
    
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Linear regression works on transformed features!")
print("By choosing the right transformation, complex relationships become linear.")

## Regularization: Controlling Complexity
**Ridge Regression** adds an L2 penalty to force small weights:

$$L = MSE + \lambda \sum w_i^2$$

Higher $\lambda$ = more penalty = simpler model (smaller weights).

In [ ]:
from sklearn.linear_model import Ridge

# Use the non-linear data with high-degree polynomial
degree = 10
poly = PolynomialFeatures(degree=degree)
X_train_poly = poly.fit_transform(X_train.reshape(-1, 1))
X_test_poly = poly.transform(X_test.reshape(-1, 1))

# Test different regularization strengths
alphas = [0, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_r2_ridge = []
test_r2_ridge = []

for alpha in alphas:
    if alpha == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=alpha)
    
    model.fit(X_train_poly, y_train)
    
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)
    
    train_r2_ridge.append(r2_score(y_train, y_train_pred))
    test_r2_ridge.append(r2_score(y_test, y_test_pred))

plt.figure(figsize=(10, 5))
plt.semilogx(alphas, train_r2_ridge, 'o-', linewidth=2, markersize=8, label='Train R²')
plt.semilogx(alphas, test_r2_ridge, 's-', linewidth=2, markersize=8, label='Test R²')
plt.xlabel('Regularization Strength (λ)')
plt.ylabel('R² Score')
plt.title('Ridge Regression: Reducing Overfitting')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

print(f"{'λ (alpha)':<12} {'Train R²':<10} {'Test R²':<10} {'Gap':<10}")
for alpha, tr2, ter2 in zip(alphas, train_r2_ridge, test_r2_ridge):
    gap = tr2 - ter2
    print(f"{alpha:<12.3f} {tr2:<10.4f} {ter2:<10.4f} {gap:<10.4f}")

print("\nNotice: Strong regularization (high λ) reduces train R² but INCREASES test R².")
print("The gap (overfitting measure) shrinks significantly!")

## Summary: Regression Decision Tree

**Step 1**: Plot your data. Is the relationship linear? Exponential? Unknown?

**Step 2**: Choose a loss function.
- **MSE**: Default. Sensitive to outliers but smooth for optimization.
- **MAE**: Robust to outliers. Harder to optimize.

**Step 3**: Choose model complexity.
- **Linear**: Fast, interpretable, but may underfit.
- **Polynomial**: Flexible, but prone to overfitting without regularization.
- **Transformed features**: Use domain knowledge to create features (log, exp, sqrt).

**Step 4**: Combat overfitting.
- **Train/test split**: Detect overfitting.
- **Regularization**: Ridge or Lasso to penalize complexity.
- **Cross-validation**: Robust evaluation.

**Step 5**: Validate assumptions.
- Residuals should be random (no patterns).
- Residuals should be normally distributed.
- No heteroscedasticity (variance shouldn't change with predicted value).

## Strengths and Weaknesses

### Strengths
- **Interpretability**: Coefficients have clear meaning (e.g., "each additional sq ft adds $X to price")
- **Simplicity**: Easy to implement, understand, and debug
- **Efficiency**: Closed-form solutions or fast optimization; scales to large datasets
- **Statistical foundations**: Confidence intervals, hypothesis tests available
- **Non-linear flexibility**: Feature engineering enables fitting complex functions

### Weaknesses
- **Assumes linear relationship**: Must transform features or lose accuracy
- **Sensitive to outliers**: MSE especially; MAE robust but harder to optimize
- **Overfitting**: High-complexity models memorize noise; regularization needed
- **Feature engineering burden**: Choosing right features/transformations requires domain knowledge
- **Causality confusion**: Correlation ≠ causation; regression finds associations, not causes
- **No learned representations**: Unlike neural networks, doesn't discover new features